In [1]:
import os
import numpy as np
import open3d as o3d
from joblib import Parallel, delayed

# =========================================================
# CONFIG (EDIT THESE PATHS)
# =========================================================
DAY_DIRS = {
    "Day1_1": r"D:\CVPR_Data\New_PLY\Day1_1",
    "Day2_1": r"D:\CVPR_Data\New_PLY\Day2_1",
    "Day3_1": r"D:\CVPR_Data\New_PLY\Day3_1",
}

VOXEL_SIZE = 0.004
NUM_POINTS = 1500
N_JOBS = 22

# ✅ NEW RULE: must have >=50 valid point clouds EACH DAY
MIN_VALID_PER_DAY = 50


# =========================================================
# HELPERS
# =========================================================
def norm_sow_id(folder_name: str) -> str:
    """If your folder names are the sow IDs, keep as-is."""
    return folder_name.strip()


def list_ply_files(folder_path: str):
    ply_files = []
    for root, _, files in os.walk(folder_path):
        for f in files:
            if f.lower().endswith(".ply"):
                ply_files.append(os.path.join(root, f))
    return ply_files


def is_valid_ply(ply_path: str) -> bool:
    """Same validity rule as training: voxel downsample then must have >= NUM_POINTS."""
    try:
        pcd = o3d.io.read_point_cloud(ply_path)
        pcd = pcd.voxel_down_sample(voxel_size=VOXEL_SIZE)
        pts = np.asarray(pcd.points)
        return pts.shape[0] >= NUM_POINTS
    except Exception:
        return False


def count_valid_in_sow_folder(sow_folder_path: str) -> int:
    """Count valid .ply files in a sow folder (including nested visit folders)."""
    ply_files = list_ply_files(sow_folder_path)
    if not ply_files:
        return 0

    flags = Parallel(n_jobs=N_JOBS, prefer="processes")(
        delayed(is_valid_ply)(p) for p in ply_files
    )
    return int(sum(flags))


# =========================================================
# MAIN
# =========================================================
def compute_stable_sows(day_dirs: dict):
    """
    Returns:
      stable_sows: list[str]
      summary_rows: list[(sow_id, day1_valid, day2_valid, day3_valid, total)]
      counts: dict[day][sow_id] = valid_count
    """
    counts = {day: {} for day in day_dirs.keys()}

    # 1) Scan all days and count valid point clouds per sow
    for day_name, day_root in day_dirs.items():
        if not os.path.isdir(day_root):
            raise FileNotFoundError(f"{day_name} path not found: {day_root}")

        sow_folders = [
            f for f in os.listdir(day_root)
            if os.path.isdir(os.path.join(day_root, f))
        ]

        print(f"\n{day_name}: found {len(sow_folders)} sow folders")

        for f in sow_folders:
            sow_id = norm_sow_id(f)
            sow_path = os.path.join(day_root, f)
            v = count_valid_in_sow_folder(sow_path)
            counts[day_name][sow_id] = v

    # 2) Keep only sows that appear in ALL 3 days
    all_days = list(day_dirs.keys())
    sows_all = set(counts[all_days[0]].keys())
    for d in all_days[1:]:
        sows_all &= set(counts[d].keys())

    # 3) Apply NEW RULE: >= MIN_VALID_PER_DAY EACH DAY
    stable_sows = []
    summary_rows = []

    for sow_id in sorted(sows_all):
        per_day = [counts[d].get(sow_id, 0) for d in all_days]
        if all(v >= MIN_VALID_PER_DAY for v in per_day):
            total = sum(per_day)
            stable_sows.append(sow_id)
            summary_rows.append((sow_id, *per_day, total))

    return stable_sows, summary_rows, counts


if __name__ == "__main__":
    stable_sows, summary_rows, counts = compute_stable_sows(DAY_DIRS)

    print("\n✅ Stable sows across ALL 3 days (>= "
          f"{MIN_VALID_PER_DAY} valid point clouds EACH DAY):")
    print(stable_sows)

    print("\nDetails: sow_id | Day1_1 | Day2_1 | Day3_1 | total")
    for row in summary_rows:
        print(row)

    # Optional: save stable list to paste into training
    print("\nPaste-ready STABLE_LIST:")
    print("STABLE_LIST =", stable_sows)


c:\Users\spaudel\AppData\Local\anaconda3\envs\tf-gpu\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.

Day1_1: found 19 sow folders

Day2_1: found 19 sow folders


KeyboardInterrupt: 

In [12]:
import os
from joblib import Parallel, delayed

# =========================================================
# CONFIG
# =========================================================
DAY_DIRS = {
    "Day1_1": r"D:\CVPR_Data\10Pig_PLY\Day1_2",
    "Day2_1": r"D:\CVPR_Data\10Pig_PLY\Day2_2",
    "Day3_1": r"D:\CVPR_Data\10Pig_PLY\Day3_2",
}

N_JOBS = 22

# ✅ NEW RULE: must have >= 50 .ply files EACH DAY that are > 1500 KB
MIN_VALID_PER_DAY = 30
MIN_SIZE_KB = 350
MIN_SIZE_BYTES = MIN_SIZE_KB * 1024


# =========================================================
# HELPERS
# =========================================================
def norm_sow_id(folder_name: str) -> str:
    """If your folder names are the sow IDs, keep as-is."""
    return folder_name.strip()


def list_ply_files(folder_path: str):
    ply_files = []
    for root, _, files in os.walk(folder_path):
        for f in files:
            if f.lower().endswith(".ply"):
                ply_files.append(os.path.join(root, f))
    return ply_files


def is_big_enough_ply(ply_path: str) -> bool:
    """Valid if file size is > MIN_SIZE_BYTES."""
    try:
        return os.path.getsize(ply_path) > MIN_SIZE_BYTES
    except OSError:
        return False


def count_big_plys_in_sow_folder(sow_folder_path: str) -> int:
    """Count .ply files in a sow folder (including nested visit folders) that are > MIN_SIZE_BYTES."""
    ply_files = list_ply_files(sow_folder_path)
    if not ply_files:
        return 0

    flags = Parallel(n_jobs=N_JOBS, prefer="processes")(
        delayed(is_big_enough_ply)(p) for p in ply_files
    )
    return int(sum(flags))


# =========================================================
# MAIN
# =========================================================
def compute_stable_sows(day_dirs: dict):
    """
    Returns:
      stable_sows: list[str]
      summary_rows: list[(sow_id, day1_big, day2_big, day3_big, total)]
      counts: dict[day][sow_id] = big_count
    """
    counts = {day: {} for day in day_dirs.keys()}

    # 1) Scan all days and count "big enough" PLYs per sow
    for day_name, day_root in day_dirs.items():
        if not os.path.isdir(day_root):
            raise FileNotFoundError(f"{day_name} path not found: {day_root}")

        sow_folders = [
            f for f in os.listdir(day_root)
            if os.path.isdir(os.path.join(day_root, f))
        ]

        print(f"\n{day_name}: found {len(sow_folders)} sow folders")

        for f in sow_folders:
            sow_id = norm_sow_id(f)
            sow_path = os.path.join(day_root, f)
            v = count_big_plys_in_sow_folder(sow_path)
            counts[day_name][sow_id] = v

    # 2) Keep only sows that appear in ALL 3 days
    all_days = list(day_dirs.keys())
    sows_all = set(counts[all_days[0]].keys())
    for d in all_days[1:]:
        sows_all &= set(counts[d].keys())

    # 3) Apply NEW RULE: >= MIN_VALID_PER_DAY EACH DAY
    stable_sows = []
    summary_rows = []

    for sow_id in sorted(sows_all):
        per_day = [counts[d].get(sow_id, 0) for d in all_days]
        if all(v >= MIN_VALID_PER_DAY for v in per_day):
            total = sum(per_day)
            stable_sows.append(sow_id)
            summary_rows.append((sow_id, *per_day, total))

    return stable_sows, summary_rows, counts


if __name__ == "__main__":
    stable_sows, summary_rows, counts = compute_stable_sows(DAY_DIRS)

    print("\n✅ Stable sows across ALL 3 days (>= "
          f"{MIN_VALID_PER_DAY} PLYs > {MIN_SIZE_KB} KB EACH DAY):")
    print(stable_sows)

    print("\nDetails: sow_id | Day1_1 | Day2_1 | Day3_1 | total")
    for row in summary_rows:
        print(row)

    print("\nPaste-ready STABLE_LIST:")
    print("STABLE_LIST =", stable_sows)



Day1_1: found 10 sow folders

Day2_1: found 10 sow folders

Day3_1: found 10 sow folders

✅ Stable sows across ALL 3 days (>= 30 PLYs > 350 KB EACH DAY):
['419', '453', '458', '467', '472', '485', '492', '495', '542-533']

Details: sow_id | Day1_1 | Day2_1 | Day3_1 | total
('419', 1155, 311, 2154, 3620)
('453', 507, 255, 38, 800)
('458', 583, 621, 608, 1812)
('467', 298, 619, 445, 1362)
('472', 1276, 1202, 1273, 3751)
('485', 1613, 3606, 1809, 7028)
('492', 533, 400, 684, 1617)
('495', 898, 487, 120, 1505)
('542-533', 548, 300, 688, 1536)

Paste-ready STABLE_LIST:
STABLE_LIST = ['419', '453', '458', '467', '472', '485', '492', '495', '542-533']
